In [1]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

72

In [3]:
index = Index(text_fields = ["content"] , keyword_fields= ["filename"])
index.fit(documents)

search_result = index.search("How does the agentic loop keep calling the model until it stops")
print(search_result[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [6]:
SYSTEM_INSTRUCTIONS = """
You are Lesson helper on the llm course.Answer only based on given context if context does not have an answer
then you should say that you don't know and do not try to answer using memory or invent something.
When you are answering the question mention the lesson you info from.
"""

class RAG:
    def __init__(self,index,config,client,model="gemini-3.5-flash"):
        self.index = index
        self.config = config
        self.client = client
        self.model = model

    def search(self,question):
        search_result = self.index.search(question)
        return search_result

    def build_context(self,search_result):
        content = []

        for lesson in search_result:
            content.append("name of the lesson" + lesson["filename"])
            content.append("\nContent:" + lesson["content"])

        return "\n".join(content).strip()

    def build_prompt(self,question,search_result):
        contents = self.build_context(search_result)
        prompt = f"""
        Question: {question}

        Context: {contents}
        """
        return prompt

    def llm(self,prompt):
        response = self.client.models.generate_content(
            model = self.model,
            contents = prompt,
            config = self.config
        )
        return response

    def rag(self,question):
        search_result = self.search(question)
        prompt = self.build_prompt(question,search_result)
        response = self.llm(prompt)
        return response.text,response.usage_metadata

In [12]:
config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTIONS,
)

client = genai.Client()

assistant = RAG(index,config,client)
question="How does the agentic loop keep calling the model until it stops?"
response = assistant.rag(question)
print(response)

('Based on the lesson **01-agentic-rag/lessons/14-agentic-loop.md** ("The Agentic Loop"), the agentic loop keeps calling the model using a `while` loop guided by a tracking flag (e.g., `has_function_calls`). \n\nSpecifically, the process works as follows:\n1. **The Loop:** The code wraps the model calls in a `while True` loop.\n2. **Checking for Function Calls:** For each response from the model, the loop checks if there are any function/tool calls requested. \n3. **Execution and Feedback:** If a function call is present, the helper executes the tool, appends the results back to the message history, and sets the `has_function_calls` flag to `True`. Because the flag is `True`, the loop runs again, sending the updated history back to the model.\n4. **The Exit Condition:** The loop stops when the model returns a final answer with no more function/tool calls. At this point, `has_function_calls` remains `False`, which triggers a `break` to exit the loop.', GenerateContentResponseUsageMetada

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents,size=2000,step=1000)
len(chunks)

295

In [16]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)


chunk_rag = RAG(index,config,client)
response = chunk_rag.rag(question)
print(response)

('Based on the lesson **01-agentic-rag/lessons/14-agentic-loop.md**, the agentic loop keeps calling the model using a `while True` loop. \n\nHere is how it works:\n1. **The Loop:** It runs a `while True` loop, sending the conversation history (including previous tool outputs) to the model.\n2. **Checking for Function Calls:** In each iteration, a flag (such as `has_function_calls`) is set to `False`. The loop then processes the model\'s response.\n3. **Executing Tools:** If the model\'s response contains a function call (`item.type == "function_call"`), the code runs the tool, appends the tool\'s output back to the message history, and sets `has_function_calls` to `True` so the loop continues.\n4. **Exit Condition:** If the model returns a final response with no function/tool calls, `has_function_calls` remains `False`, triggering a `break` to stop the loop.', GenerateContentResponseUsageMetadata(
  candidates_token_count=227,
  prompt_token_count=5373,
  prompt_tokens_details=[
    Mo

In [26]:
from pydantic_ai import Agent

def search(question : str) -> list[dict]:
    """
    Search the index in order to get relevant information about the lessons

    Args:
        question: users question in english about lessons
    """

    search_result = index.search(question)
    return search_result

SYSTEM_INSTRUCTIONS = """
You are a course teaching assistant. Answer the student's question using the search tool.Make multiple searches with different keywords before answering
"""

agent = Agent(model = "google:gemini-3.6-flash",instructions=SYSTEM_INSTRUCTIONS,tools=[search])

result = await agent.run(question)

In [27]:
result.output

'The **agentic loop** keeps calling the model until it stops by wrapping the LLM interaction in a **`while` loop** that checks whether the model wants to call a tool or provide a final answer.\n\n---\n\n### How the Loop Works\n\n1. **State Tracking (`has_function_calls`)**: \n   A boolean flag (e.g., `has_function_calls`) is initialized to `False` at the start of each iteration.\n2. **Model Call**: \n   The application sends the full message history (`messages`) along with available tools to the LLM.\n3. **Response Inspection**:\n   - **If the model outputs a function/tool call**: The code executes the function, appends the tool\'s result back to `messages`, and sets `has_function_calls = True`.\n   - **If the model outputs a text message**: The code captures the answer text for the user.\n4. **Exit Condition**: \n   At the end of the iteration, if `has_function_calls` remains `False` (meaning the model returned a final answer without requesting any tool calls), the loop breaks.\n\n---

In [28]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='How does the agentic loop keep calling the model until it stops?', timestamp=datetime.datetime(2026, 9, 22, 11, 32, 54, 177483, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 22, 11, 32, 54, 177713, tzinfo=datetime.timezone.utc), instructions="You are a course teaching assistant. Answer the student's question using the search tool.Make multiple searches with different keywords before answering", run_id='01a0c8e3-e720-71f2-abb6-5a90b8fa9674', conversation_id='01a0c8e3-e720-71f2-abb6-5a9115794dd9'),
 ModelResponse(parts=[ToolCallPart(tool_name='search', args={'question': 'how does agentic loop keep calling model until it stops'}, tool_call_id='call_160454', provider_name='google', provider_details={'thought_signature': 'EpAFCo0FAWkUfRM2w+MguUYY2HvSrB+nhZMOIrEcCu+HHdln++XCv7NTMLh49DGPLeXM01MUFiCVFV9/TKjH4c1INkC1xFuH/Gz2rU6UMNALBbHnpCr52bC628afRy7WzbslONTT7KP1OA70VRvLFL1hcCbb6VkmVX+7QMguAxhn8mgHWl+47fUeCAlXHL+tcyxb+Vne4DoX

In [29]:
result.usage

RunUsage(cost=Decimal('0.00864525'), details={'thoughts_tokens': 481, 'text_prompt_tokens': 5667}, input_text_tokens=5667, input_tokens=5667, output_reasoning_tokens=481, output_tokens=1172, requests=2, tool_calls=1)

In [30]:
result = await agent.run("How does the agentic loop work, and how is it different from plain RAG")
result.usage


RunUsage(cost=Decimal('0.00968175'), details={'thoughts_tokens': 711, 'text_prompt_tokens': 5734}, input_text_tokens=5734, input_tokens=5734, output_reasoning_tokens=711, output_tokens=1435, requests=2, tool_calls=1)